## Earthquake Data Exploration Script
Analyzes the raw earthquake dataset and generates exploration report

In [1]:
import pandas as pd
import numpy as np
import sys
from pathlib import Path

## load dataset

In [2]:
print("="*80)
print("EARTHQUAKE DATA EXPLORATION")
print("="*80)

df = pd.read_csv("../data/raw/earthquake_dataset.csv")
print(f"\n✓ Dataset loaded successfully!")
print(f"  Shape: {df.shape[0]} rows × {df.shape[1]} columns")

EARTHQUAKE DATA EXPLORATION

✓ Dataset loaded successfully!
  Shape: 50000 rows × 15 columns


## Inspect dataset structure


In [3]:
print("\n" + "="*80)
print("1. DATASET STRUCTURE")
print("="*80)

print("\nColumn Names:")
for i, col in enumerate(df.columns, 1):
  print(f"  {i:2d}. {col}")

print("\nData Types:")
print(df.dtypes)

print("\nFirst 3 Rows:")
print(df.head(3).to_string())


1. DATASET STRUCTURE

Column Names:
   1. No
   2. Earthquake code
   3. Date
   4. Time
   5. Latitude
   6. Longitude
   7. Der(km)Depth(km)
   8. xM
   9. MD
  10. ML
  11. Mw
  12. Ms
  13. Mb
  14. Kind
  15. Place

Data Types:
No                    int64
Earthquake code     float64
Date                 object
Time                 object
Latitude            float64
Longitude           float64
Der(km)Depth(km)    float64
xM                  float64
MD                  float64
ML                  float64
Mw                  float64
Ms                  float64
Mb                  float64
Kind                 object
Place                object
dtype: object

First 3 Rows:
   No  Earthquake code        Date     Time  Latitude  Longitude  Der(km)Depth(km)   xM   MD   ML   Mw   Ms   Mb Kind                                           Place
0   1     2.023070e+13  2023.07.31  53:38.3   37.4153    37.1703               4.0  3.5  0.0  3.5  3.4  0.0  0.0   Ke  DOGANLI-PAZARCIK (KAHRAMANMARAS)

## Check for data quality issues

In [4]:
print("\n" + "="*80)
print("2. DATA QUALITY ASSESSMENT")
print("="*80)

# Missing values
missing_data = pd.DataFrame({
  'Column': df.columns,
  'Missing_Count': df.isnull().sum(),
  'Missing_Percentage': (df.isnull().sum() / len(df) * 100).round(2)
})
missing_data = missing_data[missing_data['Missing_Count'] > 0]

if not missing_data.empty:
  print("\nMissing Values Found:")
  print(missing_data.to_string(index=False))
else:
  print("\nNo missing values found!")

# Duplicates
duplicates = df.duplicated().sum()
print(f"\nDuplicate Rows: {duplicates}")
if duplicates > 0:
  print(f"  Percentage: {(duplicates/len(df)*100):.2f}%")

# Zero values in magnitude columns
magnitude_cols = ['xM', 'MD', 'ML', 'Mw', 'Ms', 'Mb']
available_mag_cols = [col for col in magnitude_cols if col in df.columns]

print("\nZero Values in Magnitude Columns:")
for col in available_mag_cols:
  zero_count = (df[col] == 0).sum()
  zero_pct = (zero_count / len(df) * 100)
  print(f"  {col}: {zero_count} ({zero_pct:.1f}%)")


2. DATA QUALITY ASSESSMENT

Missing Values Found:
Column  Missing_Count  Missing_Percentage
    Mw          37078               74.16

Duplicate Rows: 0

Zero Values in Magnitude Columns:
  xM: 0 (0.0%)
  MD: 20436 (40.9%)
  ML: 28778 (57.6%)
  Mw: 2281 (4.6%)
  Ms: 49987 (100.0%)
  Mb: 49414 (98.8%)


## Generate statistical summary

In [5]:
print("\n" + "="*80)
print("3. STATISTICAL SUMMARY")
print("="*80)

print("\nNumerical Columns Statistics:")
print(df.describe().to_string())

# Magnitude analysis
magnitude_cols = ['xM', 'MD', 'ML', 'Mw', 'Ms', 'Mb']
available_mag_cols = [col for col in magnitude_cols if col in df.columns]

print("\n\nMagnitude Column Analysis:")
for col in available_mag_cols:
    non_zero = df[df[col] != 0][col]
    if len(non_zero) > 0:
        print(f"\n{col}:")
        print(f"  Total values: {len(df)}")
        print(f"  Non-zero: {len(non_zero)} ({len(non_zero)/len(df)*100:.1f}%)")
        print(f"  Range: {non_zero.min():.2f} - {non_zero.max():.2f}")
        print(f"  Mean: {non_zero.mean():.2f}")
        print(f"  Median: {non_zero.median():.2f}")
        print(f"  Std Dev: {non_zero.std():.2f}")



3. STATISTICAL SUMMARY

Numerical Columns Statistics:
                 No  Earthquake code      Latitude     Longitude  Der(km)Depth(km)            xM            MD            ML            Mw            Ms            Mb
count  50000.000000     5.000000e+04  50000.000000  50000.000000      50000.000000  50000.000000  50000.000000  50000.000000  12922.000000  50000.000000  50000.000000
mean   25000.500000     2.009235e+13     38.300424     32.963727          9.995948      3.344862      1.921914      1.464486      2.877171      0.001400      0.051334
std    14433.901067     7.962874e+10      1.464283      5.746905         12.580761      0.395971      1.613644      1.732178      1.409761      0.087321      0.472931
min        1.000000     1.994090e+13     35.000000     26.000000          0.000000      3.000000      0.000000      0.000000      0.000000      0.000000      0.000000
25%    12500.750000     2.004020e+13     37.192725     27.820000          5.000000      3.100000      0.000000

## Analyze geographic distribution

In [6]:
print("\n" + "="*80)
print("4. GEOGRAPHIC ANALYSIS")
print("="*80)

print("\nGeographic Bounds:")
print(f"  Latitude:  {df['Latitude'].min():.4f} to {df['Latitude'].max():.4f}")
print(f"  Longitude: {df['Longitude'].min():.4f} to {df['Longitude'].max():.4f}")
print(f"  Depth:     {df['Der(km)Depth(km)'].min():.1f} km to {df['Der(km)Depth(km)'].max():.1f} km")

print("\nTop 10 Locations by Earthquake Count:")
top_places = df['Place'].value_counts().head(10)
for i, (place, count) in enumerate(top_places.items(), 1):
    # Truncate long place names
    place_short = place[:60] + "..." if len(place) > 60 else place
    print(f"  {i:2d}. {place_short}: {count} earthquakes")



4. GEOGRAPHIC ANALYSIS

Geographic Bounds:
  Latitude:  35.0000 to 41.9982
  Longitude: 26.0000 to 44.9992
  Depth:     0.0 km to 154.7 km

Top 10 Locations by Earthquake Count:
   1. AKDENIZ: 4286 earthquakes
   2. EGE DENIZI: 983 earthquakes
   3. ONIKI ADALAR (AKDENIZ): 673 earthquakes
   4. GÖKOVA KÖRFEZI (AKDENIZ): 591 earthquakes
   5. GOKOVA KORFEZI (AKDENIZ): 499 earthquakes
   6. SEFERIHISAR AÇIKLARI-IZMIR (EGE DENIZI): 440 earthquakes
   7. MIDILLI ADASI (EGE DENIZI): 429 earthquakes
   8. SISAM ADASI (EGE DENIZI): 357 earthquakes
   9. KUSADASI KORFEZI (EGE DENIZI): 340 earthquakes
  10. MARMARA DENIZI: 307 earthquakes


## Analyze temporal patterns

In [7]:
print("\n" + "="*80)
print("5. TEMPORAL ANALYSIS")
print("="*80)

print("\nDate Coverage:")
print(f"  Range: {df['Date'].min()} to {df['Date'].max()}")
print(f"  Unique dates: {df['Date'].nunique()}")

print("\nTop 5 Dates by Earthquake Count:")
top_dates = df['Date'].value_counts().head(5)
for date, count in top_dates.items():
    print(f"  {date}: {count} earthquakes")



5. TEMPORAL ANALYSIS

Date Coverage:
  Range: 1994.09.11 to 2023.07.31
  Unique dates: 9639

Top 5 Dates by Earthquake Count:
  2023.02.06: 572 earthquakes
  2023.02.07: 359 earthquakes
  2023.02.08: 227 earthquakes
  2020.10.30: 186 earthquakes
  2023.02.09: 178 earthquakes


## Identify data quality issues and recommendations

In [8]:
print("\n" + "="*80)
print("6. DATA QUALITY ISSUES & CLEANING RECOMMENDATIONS")
print("="*80)

issues = []
recommendations = []

# Date format issue
sample_date = df['Date'].iloc[0]
if '.' in str(sample_date):
    issues.append("Date format uses periods (2023.07.31) instead of standard format")
    recommendations.append("Convert Date to datetime format (YYYY-MM-DD)")

# Time format
sample_time = df['Time'].iloc[0]
if ':' in str(sample_time) and len(str(sample_time).split(':')) < 3:
    issues.append("Time format appears incomplete or non-standard")
    recommendations.append("Parse and standardize Time column to HH:MM:SS format")

# Multiple magnitude columns
magnitude_cols = ['xM', 'MD', 'ML', 'Mw', 'Ms', 'Mb']
mag_count = sum(1 for col in magnitude_cols if col in df.columns)
if mag_count > 1:
    issues.append(f"Multiple magnitude columns ({mag_count}) with many zero values")
    recommendations.append("Create unified magnitude column (priority: Mw > ML > MD > xM)")

# Column naming
if 'Der(km)Depth(km)' in df.columns:
    issues.append("Non-standard column naming: 'Der(km)Depth(km)'")
    recommendations.append("Rename columns to snake_case for database compatibility")

# Scientific notation
if df['Earthquake code'].dtype == 'float64':
    issues.append("Earthquake code stored as float (scientific notation)")
    recommendations.append("Convert earthquake_code to string or integer")

print("\nIssues Found:")
for i, issue in enumerate(issues, 1):
    print(f"  {i}. {issue}")

print("\nRecommended Cleaning Steps:")
for i, rec in enumerate(recommendations, 1):
    print(f"  {i}. {rec}")

print("\n  6. Extract region/province from Place column")
print("  7. Create magnitude categories (Minor, Light, Moderate, Strong, Major)")
print("  8. Create temporal features (year, month, day, hour)")
print("  9. Handle missing and zero values appropriately")



6. DATA QUALITY ISSUES & CLEANING RECOMMENDATIONS

Issues Found:
  1. Date format uses periods (2023.07.31) instead of standard format
  2. Time format appears incomplete or non-standard
  3. Multiple magnitude columns (6) with many zero values
  4. Non-standard column naming: 'Der(km)Depth(km)'
  5. Earthquake code stored as float (scientific notation)

Recommended Cleaning Steps:
  1. Convert Date to datetime format (YYYY-MM-DD)
  2. Parse and standardize Time column to HH:MM:SS format
  3. Create unified magnitude column (priority: Mw > ML > MD > xM)
  4. Rename columns to snake_case for database compatibility
  5. Convert earthquake_code to string or integer

  6. Extract region/province from Place column
  7. Create magnitude categories (Minor, Light, Moderate, Strong, Major)
  8. Create temporal features (year, month, day, hour)
  9. Handle missing and zero values appropriately


## Generate final summary

In [9]:
print("\n" + "="*80)
print("7. SUMMARY")
print("="*80)

magnitude_cols = [col for col in ['xM', 'MD', 'ML', 'Mw', 'Ms', 'Mb'] if col in df.columns]

# Calculate max magnitude from available columns
max_magnitudes = []
for col in magnitude_cols:
    non_zero = df[df[col] != 0][col]
    if len(non_zero) > 0:
        max_magnitudes.append(non_zero.max())

max_mag = max(max_magnitudes) if max_magnitudes else 0

print(f"\nDataset Overview:")
print(f"  Total Earthquakes: {len(df):,}")
print(f"  Unique Locations: {df['Place'].nunique():,}")
print(f"  Date Range: {df['Date'].min()} to {df['Date'].max()}")
print(f"  Geographic Coverage: Turkey and surrounding regions")
print(f"  Magnitude Range: 0 to {max_mag:.1f}")
print(f"  Depth Range: {df['Der(km)Depth(km)'].min():.1f} to {df['Der(km)Depth(km)'].max():.1f} km")



7. SUMMARY



Dataset Overview:
  Total Earthquakes: 50,000
  Unique Locations: 33,489
  Date Range: 1994.09.11 to 2023.07.31
  Geographic Coverage: Turkey and surrounding regions
  Magnitude Range: 0 to 7.7
  Depth Range: 0.0 to 154.7 km
